In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def csv_to_tif_with_attributes(input_csv, output_folder, pixel_size=10):
    """
    将CSV文件转换为TIF栅格文件，并为所有字段创建编码表
    Time字段被删除，其他所有字段都进行编码保留
    
    参数:
    input_csv: 输入CSV文件路径
    output_folder: 输出文件夹路径
    pixel_size: 像素大小（米）
    """
    # 读取CSV文件
    df = pd.read_csv(input_csv)
    print(f"正在处理文件: {input_csv}")
    print(f"CSV文件形状: {df.shape}")
    print(f"列名: {list(df.columns)}")
    
    # 确保输出文件夹存在
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # 识别经纬度列
    lat_col = 'latitude'
    lon_col = 'longitude'
    
    # 检查经纬度列是否存在
    if lat_col not in df.columns or lon_col not in df.columns:
        print(f"错误: 缺少必要的经纬度列 ({lat_col}, {lon_col})")
        return False
    
    # 删除Time列
    if 'Time' in df.columns:
        df = df.drop(columns=['Time'])
        print("已删除Time列")
    
    # 检查坐标是否为数值类型
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # 获取除经纬度外的所有其他列
    attribute_cols = [col for col in df.columns if col not in [lat_col, lon_col]]
    print(f"保留的属性列: {attribute_cols}")
    
    # 创建编码字典
    encoding_dict = {}
    
    # 处理所有非经纬度列，创建编码
    for col in attribute_cols:
        unique_values = df[col].dropna().unique()
        value_to_code = {val: idx for idx, val in enumerate(unique_values)}
        encoding_dict[col] = value_to_code
        
        # 将原列转换为数值编码
        df[col] = df[col].map(value_to_code)
    
    # 创建编码表CSV文件
    if encoding_dict:
        encoding_csv_path = os.path.join(output_folder, f"{Path(input_csv).stem}_encoding.csv")
        encoding_data = []
        for col, mapping in encoding_dict.items():
            for original_value, code in mapping.items():
                encoding_data.append({
                    'Column': col,
                    'Original_Value': original_value,
                    'Encoded_Value': code
                })
        
        encoding_df = pd.DataFrame(encoding_data)
        encoding_df.to_csv(encoding_csv_path, index=False)
        print(f"编码表已保存: {encoding_csv_path}")
    
    # 选择一个字段作为栅格值（这里选择第一个可用字段，也可以选择其他字段）
    # 为了创建栅格，我们需要一个数值字段，所以使用第一个属性列
    if attribute_cols:
        value_col = attribute_cols[0]
        print(f"使用列 '{value_col}' 作为栅格值")
    else:
        print(f"错误: 没有可用的属性列")
        return False
    
    # 清理数据，移除无效值
    df_clean = df.dropna(subset=[lat_col, lon_col, value_col])
    
    if df_clean.empty:
        print(f"错误: 处理后没有有效数据")
        return False
    
    print(f"处理后的有效数据点数量: {len(df_clean)}")
    
    # 计算坐标范围
    min_lon, max_lon = df_clean[lon_col].min(), df_clean[lon_col].max()
    min_lat, max_lat = df_clean[lat_col].min(), df_clean[lat_col].max()
    
    print(f"坐标范围 - 经度: {min_lon:.6f} 到 {max_lon:.6f}, 纬度: {min_lat:.6f} 到 {max_lat:.6f}")
    
    # 计算经纬度差值（用于10米像素）
    lat_factor = 111320 * np.cos(np.radians((min_lat + max_lat) / 2))
    lon_per_pixel = pixel_size / lat_factor
    lat_per_pixel = pixel_size / 110574
    
    # 计算栅格尺寸
    width = int((max_lon - min_lon) / lon_per_pixel) + 1
    height = int((max_lat - min_lat) / lat_per_pixel) + 1
    
    print(f"栅格尺寸: {width} x {height}")
    print(f"像素分辨率 - 经度方向: {lon_per_pixel:.8f} 度, 纬度方向: {lat_per_pixel:.8f} 度")
    
    # 创建栅格数组
    raster_array = np.full((height, width), np.nan, dtype=np.float32)
    
    # 将点数据映射到栅格
    for _, row in df_clean.iterrows():
        lon, lat, value = row[lon_col], row[lat_col], row[value_col]
        
        # 计算在栅格中的行列索引
        col = int((lon - min_lon) / lon_per_pixel)
        row_idx = int((max_lat - lat) / lat_per_pixel)  # 注意Y轴方向
        
        # 检查索引是否在范围内
        if 0 <= col < width and 0 <= row_idx < height:
            # 如果该位置已有值，可以选择覆盖或跳过
            # 这里选择覆盖
            raster_array[row_idx, col] = value
    
    # 设置仿射变换参数
    transform = from_origin(min_lon, max_lat, lon_per_pixel, lat_per_pixel)
    
    # 生成输出文件名
    base_name = Path(input_csv).stem
    output_tif = os.path.join(output_folder, f"{base_name}_grid.tif")
    
    # 创建TIF文件（使用WGS84坐标系统）
    with rasterio.open(
        output_tif,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=raster_array.dtype,
        crs='EPSG:4326',  # WGS84坐标系统
        transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(raster_array, 1)
    
    print(f"成功创建栅格文件: {output_tif}")
    print(f"栅格统计信息:")
    print(f"  - 有效值数量: {np.count_nonzero(~np.isnan(raster_array))}")
    if not np.all(np.isnan(raster_array)):
        print(f"  - 最小值: {np.nanmin(raster_array):.4f}")
        print(f"  - 最大值: {np.nanmax(raster_array):.4f}")
        print(f"  - 平均值: {np.nanmean(raster_array):.4f}")
    else:
        print(f"  - 栅格中没有有效数值")
    
    # 保存处理后的数据（包含编码后的所有字段）
    processed_csv_path = os.path.join(output_folder, f"{base_name}_processed.csv")
    df_clean.to_csv(processed_csv_path, index=False)
    print(f"处理后的数据已保存: {processed_csv_path}")
    
    return True

# 设置输入文件路径
input_csv = r"E:\chuli\ratio_to_tif\hongkong_accidents.csv"
output_folder = r"E:\chuli\ratio_to_tif"

# 执行转换
if os.path.exists(input_csv):
    success = csv_to_tif_with_attributes(input_csv, output_folder, pixel_size=10)
    if success:
        print(f"\n转换完成！")
        print(f"输出文件:")
        print(f"  - 栅格文件: {Path(input_csv).stem}_grid.tif")
        print(f"  - 编码表: {Path(input_csv).stem}_encoding.csv")
        print(f"  - 处理后数据: {Path(input_csv).stem}_processed.csv")
    else:
        print("转换失败！")
else:
    print(f"输入文件不存在: {input_csv}")

# 验证生成的TIF文件
print("\n验证生成的TIF文件:")
tif_files = list(Path(output_folder).glob(f"{Path(input_csv).stem}_grid.tif"))
for tif_file in tif_files:
    try:
        with rasterio.open(tif_file) as src:
            print(f"文件: {tif_file.name}")
            print(f"  - 尺寸: {src.width} x {src.height}")
            print(f"  - 波段数: {src.count}")
            print(f"  - 数据类型: {src.dtypes[0]}")
            print(f"  - 坐标系: {src.crs}")
            print(f"  - 范围: {src.bounds}")
            print("-" * 30)
    except Exception as e:
        print(f"读取文件 {tif_file} 时出错: {str(e)}")

# 显示编码表内容
encoding_file = os.path.join(output_folder, f"{Path(input_csv).stem}_encoding.csv")
if os.path.exists(encoding_file):
    print(f"\n编码表内容:")
    encoding_df = pd.read_csv(encoding_file)
    print(encoding_df)
else:
    print(f"\n未生成编码表（可能没有字符串字段需要编码）")

# 显示处理后的数据预览
processed_file = os.path.join(output_folder, f"{Path(input_csv).stem}_processed.csv")
if os.path.exists(processed_file):
    print(f"\n处理后数据预览:")
    processed_df = pd.read_csv(processed_file)
    print(processed_df.head())
    print(f"处理后数据形状: {processed_df.shape}")


In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def csv_to_tif_with_attributes(input_csv, output_folder, pixel_size=10):
    """
    将CSV文件转换为TIF栅格文件，并为所有字段创建编码表
    OFNS_DESC字段被删除，其他所有字段都进行编码保留
    
    参数:
    input_csv: 输入CSV文件路径
    output_folder: 输出文件夹路径
    pixel_size: 像素大小（米）
    """
    # 读取CSV文件
    df = pd.read_csv(input_csv)
    print(f"正在处理文件: {input_csv}")
    print(f"CSV文件形状: {df.shape}")
    print(f"列名: {list(df.columns)}")
    
    # 确保输出文件夹存在
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # 识别经纬度列
    lat_col = 'Latitude'
    lon_col = 'Longitude'
    
    # 检查经纬度列是否存在
    if lat_col not in df.columns or lon_col not in df.columns:
        print(f"错误: 缺少必要的经纬度列 ({lat_col}, {lon_col})")
        return False
    
    # 删除OFNS_DESC列
    if 'OFNS_DESC' in df.columns:
        df = df.drop(columns=['OFNS_DESC'])
        print("已删除OFNS_DESC列")
    
    # 检查坐标是否为数值类型
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # 获取除经纬度外的所有其他列
    attribute_cols = [col for col in df.columns if col not in [lat_col, lon_col]]
    print(f"保留的属性列: {attribute_cols}")
    
    # 创建编码字典
    encoding_dict = {}
    
    # 处理所有非经纬度列，创建编码
    for col in attribute_cols:
        unique_values = df[col].dropna().unique()
        value_to_code = {val: idx for idx, val in enumerate(unique_values)}
        encoding_dict[col] = value_to_code
        
        # 将原列转换为数值编码
        df[col] = df[col].map(value_to_code)
    
    # 创建编码表CSV文件
    if encoding_dict:
        encoding_csv_path = os.path.join(output_folder, f"{Path(input_csv).stem}_encoding.csv")
        encoding_data = []
        for col, mapping in encoding_dict.items():
            for original_value, code in mapping.items():
                encoding_data.append({
                    'Column': col,
                    'Original_Value': original_value,
                    'Encoded_Value': code
                })
        
        encoding_df = pd.DataFrame(encoding_data)
        encoding_df.to_csv(encoding_csv_path, index=False)
        print(f"编码表已保存: {encoding_csv_path}")
    
    # 选择一个字段作为栅格值（这里选择第一个可用字段）
    if attribute_cols:
        value_col = attribute_cols[0]
        print(f"使用列 '{value_col}' 作为栅格值")
    else:
        print(f"错误: 没有可用的属性列")
        return False
    
    # 清理数据，移除无效值
    df_clean = df.dropna(subset=[lat_col, lon_col, value_col])
    
    if df_clean.empty:
        print(f"错误: 处理后没有有效数据")
        return False
    
    print(f"处理后的有效数据点数量: {len(df_clean)}")
    
    # 计算坐标范围
    min_lon, max_lon = df_clean[lon_col].min(), df_clean[lon_col].max()
    min_lat, max_lat = df_clean[lat_col].min(), df_clean[lat_col].max()
    
    print(f"坐标范围 - 经度: {min_lon:.6f} 到 {max_lon:.6f}, 纬度: {min_lat:.6f} 到 {max_lat:.6f}")
    
    # 计算经纬度差值（用于10米像素）
    lat_factor = 111320 * np.cos(np.radians((min_lat + max_lat) / 2))
    lon_per_pixel = pixel_size / lat_factor
    lat_per_pixel = pixel_size / 110574
    
    # 计算栅格尺寸
    width = int((max_lon - min_lon) / lon_per_pixel) + 1
    height = int((max_lat - min_lat) / lat_per_pixel) + 1
    
    print(f"栅格尺寸: {width} x {height}")
    print(f"像素分辨率 - 经度方向: {lon_per_pixel:.8f} 度, 纬度方向: {lat_per_pixel:.8f} 度")
    
    # 创建栅格数组
    raster_array = np.full((height, width), np.nan, dtype=np.float32)
    
    # 将点数据映射到栅格
    for _, row in df_clean.iterrows():
        lon, lat, value = row[lon_col], row[lat_col], row[value_col]
        
        # 计算在栅格中的行列索引
        col = int((lon - min_lon) / lon_per_pixel)
        row_idx = int((max_lat - lat) / lat_per_pixel)  # 注意Y轴方向
        
        # 检查索引是否在范围内
        if 0 <= col < width and 0 <= row_idx < height:
            # 如果该位置已有值，可以选择覆盖或跳过
            # 这里选择覆盖
            raster_array[row_idx, col] = value
    
    # 设置仿射变换参数
    transform = from_origin(min_lon, max_lat, lon_per_pixel, lat_per_pixel)
    
    # 生成输出文件名
    base_name = Path(input_csv).stem
    output_tif = os.path.join(output_folder, f"{base_name}_grid.tif")
    
    # 创建TIF文件（使用WGS84坐标系统）
    with rasterio.open(
        output_tif,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=raster_array.dtype,
        crs='EPSG:4326',  # WGS84坐标系统
        transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(raster_array, 1)
    
    print(f"成功创建栅格文件: {output_tif}")
    print(f"栅格统计信息:")
    print(f"  - 有效值数量: {np.count_nonzero(~np.isnan(raster_array))}")
    if not np.all(np.isnan(raster_array)):
        print(f"  - 最小值: {np.nanmin(raster_array):.4f}")
        print(f"  - 最大值: {np.nanmax(raster_array):.4f}")
        print(f"  - 平均值: {np.nanmean(raster_array):.4f}")
    else:
        print(f"  - 栅格中没有有效数值")
    
    # 保存处理后的数据（包含编码后的所有字段）
    processed_csv_path = os.path.join(output_folder, f"{base_name}_processed.csv")
    df_clean.to_csv(processed_csv_path, index=False)
    print(f"处理后的数据已保存: {processed_csv_path}")
    
    return True

# 设置输入文件路径
input_csv = r"E:\chuli\ratio_to_tif\newyork_arrests.csv"
output_folder = r"E:\chuli\ratio_to_tif"

# 执行转换
if os.path.exists(input_csv):
    success = csv_to_tif_with_attributes(input_csv, output_folder, pixel_size=10)
    if success:
        print(f"\n转换完成！")
        print(f"输出文件:")
        print(f"  - 栅格文件: {Path(input_csv).stem}_grid.tif")
        print(f"  - 编码表: {Path(input_csv).stem}_encoding.csv")
        print(f"  - 处理后数据: {Path(input_csv).stem}_processed.csv")
    else:
        print("转换失败！")
else:
    print(f"输入文件不存在: {input_csv}")

# 验证生成的TIF文件
print("\n验证生成的TIF文件:")
tif_files = list(Path(output_folder).glob(f"{Path(input_csv).stem}_grid.tif"))
for tif_file in tif_files:
    try:
        with rasterio.open(tif_file) as src:
            print(f"文件: {tif_file.name}")
            print(f"  - 尺寸: {src.width} x {src.height}")
            print(f"  - 波段数: {src.count}")
            print(f"  - 数据类型: {src.dtypes[0]}")
            print(f"  - 坐标系: {src.crs}")
            print(f"  - 范围: {src.bounds}")
            print("-" * 30)
    except Exception as e:
        print(f"读取文件 {tif_file} 时出错: {str(e)}")

# 显示编码表内容
encoding_file = os.path.join(output_folder, f"{Path(input_csv).stem}_encoding.csv")
if os.path.exists(encoding_file):
    print(f"\n编码表内容:")
    encoding_df = pd.read_csv(encoding_file)
    print(encoding_df)
else:
    print(f"\n未生成编码表（可能没有字符串字段需要编码）")

# 显示处理后的数据预览
processed_file = os.path.join(output_folder, f"{Path(input_csv).stem}_processed.csv")
if os.path.exists(processed_file):
    print(f"\n处理后数据预览:")
    processed_df = pd.read_csv(processed_file)
    print(processed_df.head())
    print(f"处理后数据形状: {processed_df.shape}")


In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def csv_to_tif_multiple_attributes(input_csv, output_folder, pixel_size=10):
    """
    将CSV文件转换为多个TIF栅格文件
    date和ratio_svid字段被删除，其他所有数值字段都输出为独立的栅格文件
    
    参数:
    input_csv: 输入CSV文件路径
    output_folder: 输出文件夹路径
    pixel_size: 像素大小（米）
    """
    # 读取CSV文件
    df = pd.read_csv(input_csv)
    print(f"正在处理文件: {input_csv}")
    print(f"CSV文件形状: {df.shape}")
    print(f"列名: {list(df.columns)}")
    
    # 确保输出文件夹存在
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # 识别经纬度列
    lat_col = 'latitude'
    lon_col = 'longitude'
    
    # 检查经纬度列是否存在
    if lat_col not in df.columns or lon_col not in df.columns:
        print(f"错误: 缺少必要的经纬度列 ({lat_col}, {lon_col})")
        return False
    
    # 删除date和ratio_svid列
    columns_to_drop = []
    if 'date' in df.columns:
        columns_to_drop.append('date')
    if 'ratio_svid' in df.columns:
        columns_to_drop.append('ratio_svid')
    
    if columns_to_drop:
        df = df.drop(columns=columns_to_drop)
        print(f"已删除列: {columns_to_drop}")
    
    # 检查坐标是否为数值类型
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # 获取除经纬度外的所有其他列（这些应该是数值列）
    attribute_cols = [col for col in df.columns if col not in [lat_col, lon_col]]
    print(f"保留的属性列: {attribute_cols}")
    
    # 检查所有保留的列是否为数值类型
    numeric_cols = []
    for col in attribute_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].dropna().size > 0:  # 如果有有效数值
            numeric_cols.append(col)
    
    print(f"有效的数值列: {numeric_cols}")
    
    if not numeric_cols:
        print("错误: 没有找到有效的数值列")
        return False
    
    # 清理数据，移除无效值
    all_cols_needed = [lat_col, lon_col] + numeric_cols
    df_clean = df.dropna(subset=all_cols_needed)
    
    if df_clean.empty:
        print(f"错误: 处理后没有有效数据")
        return False
    
    print(f"处理后的有效数据点数量: {len(df_clean)}")
    
    # 计算坐标范围
    min_lon, max_lon = df_clean[lon_col].min(), df_clean[lon_col].max()
    min_lat, max_lat = df_clean[lat_col].min(), df_clean[lat_col].max()
    
    print(f"坐标范围 - 经度: {min_lon:.6f} 到 {max_lon:.6f}, 纬度: {min_lat:.6f} 到 {max_lat:.6f}")
    
    # 计算经纬度差值（用于10米像素）
    lat_factor = 111320 * np.cos(np.radians((min_lat + max_lat) / 2))
    lon_per_pixel = pixel_size / lat_factor
    lat_per_pixel = pixel_size / 110574
    
    # 计算栅格尺寸
    width = int((max_lon - min_lon) / lon_per_pixel) + 1
    height = int((max_lat - min_lat) / lat_per_pixel) + 1
    
    print(f"栅格尺寸: {width} x {height}")
    print(f"像素分辨率 - 经度方向: {lon_per_pixel:.8f} 度, 纬度方向: {lat_per_pixel:.8f} 度")
    
    # 为每个数值列创建单独的栅格文件
    success_count = 0
    for value_col in numeric_cols:
        print(f"\n处理列: {value_col}")
        
        # 创建栅格数组
        raster_array = np.full((height, width), np.nan, dtype=np.float32)
        
        # 将点数据映射到栅格
        for _, row in df_clean.iterrows():
            lon, lat, value = row[lon_col], row[lat_col], row[value_col]
            
            # 计算在栅格中的行列索引
            col = int((lon - min_lon) / lon_per_pixel)
            row_idx = int((max_lat - lat) / lat_per_pixel)  # 注意Y轴方向
            
            # 检查索引是否在范围内
            if 0 <= col < width and 0 <= row_idx < height:
                # 如果该位置已有值，可以选择覆盖或跳过
                # 这里选择覆盖
                raster_array[row_idx, col] = value
        
        # 设置仿射变换参数
        transform = from_origin(min_lon, max_lat, lon_per_pixel, lat_per_pixel)
        
        # 生成输出文件名
        base_name = Path(input_csv).stem
        output_tif = os.path.join(output_folder, f"{base_name}_{value_col}.tif")
        
        # 创建TIF文件（使用WGS84坐标系统）
        with rasterio.open(
            output_tif,
            'w',
            driver='GTiff',
            height=height,
            width=width,
            count=1,
            dtype=raster_array.dtype,
            crs='EPSG:4326',  # WGS84坐标系统
            transform=transform,
            nodata=np.nan
        ) as dst:
            dst.write(raster_array, 1)
        
        print(f"成功创建栅格文件: {output_tif}")
        print(f"栅格统计信息:")
        print(f"  - 有效值数量: {np.count_nonzero(~np.isnan(raster_array))}")
        if not np.all(np.isnan(raster_array)):
            print(f"  - 最小值: {np.nanmin(raster_array):.6f}")
            print(f"  - 最大值: {np.nanmax(raster_array):.6f}")
            print(f"  - 平均值: {np.nanmean(raster_array):.6f}")
        else:
            print(f"  - 栅格中没有有效数值")
        
        success_count += 1
    
    # 保存处理后的数据（包含所有保留的列）
    processed_csv_path = os.path.join(output_folder, f"{base_name}_processed.csv")
    df_clean.to_csv(processed_csv_path, index=False)
    print(f"\n处理后的数据已保存: {processed_csv_path}")
    
    print(f"\n总共成功创建了 {success_count} 个栅格文件")
    
    return True

# 设置输入文件路径
input_csv = r"E:\chuli\ratio_to_tif\ratio_HK_individual.scv"  # 注意：您写的是.scv，应该是.csv
output_folder = r"E:\chuli\ratio_to_tif"

# 检查文件是否存在，如果.scv不存在则尝试.csv
if not os.path.exists(input_csv):
    input_csv = r"E:\chuli\ratio_to_tif\ratio_HK_individual.csv"
    if not os.path.exists(input_csv):
        print(f"输入文件不存在: {input_csv}")
        print(f"也尝试了: E:\chuli\ratio_to_tif\ratio_HK_individual.csv")
    else:
        print(f"找到文件: {input_csv}")
else:
    print(f"找到文件: {input_csv}")

# 执行转换
if os.path.exists(input_csv):
    success = csv_to_tif_multiple_attributes(input_csv, output_folder, pixel_size=10)
    if success:
        print(f"\n转换完成！")
        print(f"输出文件保存在: {output_folder}")
    else:
        print("转换失败！")
else:
    print(f"输入文件不存在: {input_csv}")

# 验证生成的TIF文件
print("\n验证生成的TIF文件:")
tif_files = list(Path(output_folder).glob(f"{Path(input_csv).stem}_*.tif"))
tif_files = [f for f in tif_files if f.name != f"{Path(input_csv).stem}_processed.csv".replace('_processed.csv', '.tif')]
for tif_file in tif_files:
    try:
        with rasterio.open(tif_file) as src:
            print(f"文件: {tif_file.name}")
            print(f"  - 尺寸: {src.width} x {src.height}")
            print(f"  - 波段数: {src.count}")
            print(f"  - 数据类型: {src.dtypes[0]}")
            print(f"  - 坐标系: {src.crs}")
            print(f"  - 范围: {src.bounds}")
            print("-" * 30)
    except Exception as e:
        print(f"读取文件 {tif_file} 时出错: {str(e)}")

# 显示处理后的数据预览
processed_file = os.path.join(output_folder, f"{Path(input_csv).stem}_processed.csv")
if os.path.exists(processed_file):
    print(f"\n处理后数据预览:")
    processed_df = pd.read_csv(processed_file)
    print(processed_df.head())
    print(f"处理后数据形状: {processed_df.shape}")
    print(f"列名: {list(processed_df.columns)}")


In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def csv_to_tif_multiple_attributes(input_csv, output_folder, pixel_size=10):
    """
    将CSV文件转换为多个TIF栅格文件
    date和ratio_svid字段被删除，其他所有数值字段都输出为独立的栅格文件
    
    参数:
    input_csv: 输入CSV文件路径
    output_folder: 输出文件夹路径
    pixel_size: 像素大小（米）
    """
    # 读取CSV文件
    df = pd.read_csv(input_csv)
    print(f"正在处理文件: {input_csv}")
    print(f"CSV文件形状: {df.shape}")
    print(f"列名: {list(df.columns)}")
    
    # 确保输出文件夹存在
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # 识别经纬度列
    lat_col = 'latitude'
    lon_col = 'longitude'
    
    # 检查经纬度列是否存在
    if lat_col not in df.columns or lon_col not in df.columns:
        print(f"错误: 缺少必要的经纬度列 ({lat_col}, {lon_col})")
        return False
    
    # 删除date和ratio_svid列
    columns_to_drop = []
    if 'date' in df.columns:
        columns_to_drop.append('date')
    if 'ratio_svid' in df.columns:
        columns_to_drop.append('ratio_svid')
    
    if columns_to_drop:
        df = df.drop(columns=columns_to_drop)
        print(f"已删除列: {columns_to_drop}")
    
    # 检查坐标是否为数值类型
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # 获取除经纬度外的所有其他列（这些应该是数值列）
    attribute_cols = [col for col in df.columns if col not in [lat_col, lon_col]]
    print(f"保留的属性列: {attribute_cols}")
    
    # 检查所有保留的列是否为数值类型
    numeric_cols = []
    for col in attribute_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].dropna().size > 0:  # 如果有有效数值
            numeric_cols.append(col)
    
    print(f"有效的数值列: {numeric_cols}")
    
    if not numeric_cols:
        print("错误: 没有找到有效的数值列")
        return False
    
    # 清理数据，移除无效值
    all_cols_needed = [lat_col, lon_col] + numeric_cols
    df_clean = df.dropna(subset=all_cols_needed)
    
    if df_clean.empty:
        print(f"错误: 处理后没有有效数据")
        return False
    
    print(f"处理后的有效数据点数量: {len(df_clean)}")
    
    # 计算坐标范围
    min_lon, max_lon = df_clean[lon_col].min(), df_clean[lon_col].max()
    min_lat, max_lat = df_clean[lat_col].min(), df_clean[lat_col].max()
    
    print(f"坐标范围 - 经度: {min_lon:.6f} 到 {max_lon:.6f}, 纬度: {min_lat:.6f} 到 {max_lat:.6f}")
    
    # 计算经纬度差值（用于10米像素）
    lat_factor = 111320 * np.cos(np.radians((min_lat + max_lat) / 2))
    lon_per_pixel = pixel_size / lat_factor
    lat_per_pixel = pixel_size / 110574
    
    # 计算栅格尺寸
    width = int((max_lon - min_lon) / lon_per_pixel) + 1
    height = int((max_lat - min_lat) / lat_per_pixel) + 1
    
    print(f"栅格尺寸: {width} x {height}")
    print(f"像素分辨率 - 经度方向: {lon_per_pixel:.8f} 度, 纬度方向: {lat_per_pixel:.8f} 度")
    
    # 为每个数值列创建单独的栅格文件
    success_count = 0
    for value_col in numeric_cols:
        print(f"\n处理列: {value_col}")
        
        # 创建栅格数组
        raster_array = np.full((height, width), np.nan, dtype=np.float32)
        
        # 将点数据映射到栅格
        for _, row in df_clean.iterrows():
            lon, lat, value = row[lon_col], row[lat_col], row[value_col]
            
            # 计算在栅格中的行列索引
            col = int((lon - min_lon) / lon_per_pixel)
            row_idx = int((max_lat - lat) / lat_per_pixel)  # 注意Y轴方向
            
            # 检查索引是否在范围内
            if 0 <= col < width and 0 <= row_idx < height:
                # 如果该位置已有值，可以选择覆盖或跳过
                # 这里选择覆盖
                raster_array[row_idx, col] = value
        
        # 设置仿射变换参数
        transform = from_origin(min_lon, max_lat, lon_per_pixel, lat_per_pixel)
        
        # 生成输出文件名
        base_name = Path(input_csv).stem
        output_tif = os.path.join(output_folder, f"{base_name}_{value_col}.tif")
        
        # 创建TIF文件（使用WGS84坐标系统）
        with rasterio.open(
            output_tif,
            'w',
            driver='GTiff',
            height=height,
            width=width,
            count=1,
            dtype=raster_array.dtype,
            crs='EPSG:4326',  # WGS84坐标系统
            transform=transform,
            nodata=np.nan
        ) as dst:
            dst.write(raster_array, 1)
        
        print(f"成功创建栅格文件: {output_tif}")
        print(f"栅格统计信息:")
        print(f"  - 有效值数量: {np.count_nonzero(~np.isnan(raster_array))}")
        if not np.all(np.isnan(raster_array)):
            print(f"  - 最小值: {np.nanmin(raster_array):.6f}")
            print(f"  - 最大值: {np.nanmax(raster_array):.6f}")
            print(f"  - 平均值: {np.nanmean(raster_array):.6f}")
        else:
            print(f"  - 栅格中没有有效数值")
        
        success_count += 1
    
    # 保存处理后的数据（包含所有保留的列）
    processed_csv_path = os.path.join(output_folder, f"{base_name}_processed.csv")
    df_clean.to_csv(processed_csv_path, index=False)
    print(f"\n处理后的数据已保存: {processed_csv_path}")
    
    print(f"\n总共成功创建了 {success_count} 个栅格文件")
    
    return True

# 设置输入文件路径
input_csv = r"E:\chuli\ratio_to_tif\ratio_HK_plus.csv"
output_folder = r"E:\chuli\ratio_to_tif"

# 执行转换
if os.path.exists(input_csv):
    success = csv_to_tif_multiple_attributes(input_csv, output_folder, pixel_size=10)
    if success:
        print(f"\n转换完成！")
        print(f"输出文件保存在: {output_folder}")
        print(f"创建的栅格文件:")
        for col in ['Enclosure_degree', 'Openness', 'Protection', 'Surveillance_and_lighting', 
                    'Street_furniture', 'Greening_degree', 'Traffic_flow', 'Pedestrian_density']:
            tif_file = os.path.join(output_folder, f"ratio_HK_plus_{col}.tif")
            if os.path.exists(tif_file):
                print(f"  - {col}.tif")
    else:
        print("转换失败！")
else:
    print(f"输入文件不存在: {input_csv}")

# 验证生成的TIF文件
print("\n验证生成的TIF文件:")
tif_files = list(Path(output_folder).glob(f"ratio_HK_plus_*.tif"))
tif_files = [f for f in tif_files if not f.name.endswith('_processed.csv.tif')]
for tif_file in tif_files:
    try:
        with rasterio.open(tif_file) as src:
            print(f"文件: {tif_file.name}")
            print(f"  - 尺寸: {src.width} x {src.height}")
            print(f"  - 波段数: {src.count}")
            print(f"  - 数据类型: {src.dtypes[0]}")
            print(f"  - 坐标系: {src.crs}")
            print(f"  - 范围: {src.bounds}")
            print("-" * 30)
    except Exception as e:
        print(f"读取文件 {tif_file} 时出错: {str(e)}")

# 显示处理后的数据预览
processed_file = os.path.join(output_folder, f"ratio_HK_plus_processed.csv")
if os.path.exists(processed_file):
    print(f"\n处理后数据预览:")
    processed_df = pd.read_csv(processed_file)
    print(processed_df.head())
    print(f"处理后数据形状: {processed_df.shape}")
    print(f"列名: {list(processed_df.columns)}")


In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def csv_to_tif_multiple_attributes(input_csv, output_folder, pixel_size=10):
    """
    将CSV文件转换为多个TIF栅格文件
    date和ratio_svid字段被删除，其他所有数值字段都输出为独立的栅格文件
    
    参数:
    input_csv: 输入CSV文件路径
    output_folder: 输出文件夹路径
    pixel_size: 像素大小（米）
    """
    # 读取CSV文件
    df = pd.read_csv(input_csv)
    print(f"正在处理文件: {input_csv}")
    print(f"CSV文件形状: {df.shape}")
    print(f"列名: {list(df.columns)}")
    
    # 确保输出文件夹存在
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # 识别经纬度列
    lat_col = 'lat'
    lon_col = 'lon'
    
    # 检查经纬度列是否存在
    if lat_col not in df.columns or lon_col not in df.columns:
        print(f"错误: 缺少必要的经纬度列 ({lat_col}, {lon_col})")
        return False
    
    # 删除date和ratio_svid列
    columns_to_drop = []
    if 'date' in df.columns:
        columns_to_drop.append('date')
    if 'ratio_svid' in df.columns:
        columns_to_drop.append('ratio_svid')
    
    if columns_to_drop:
        df = df.drop(columns=columns_to_drop)
        print(f"已删除列: {columns_to_drop}")
    
    # 检查坐标是否为数值类型
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # 获取除经纬度外的所有其他列（这些应该是数值列）
    attribute_cols = [col for col in df.columns if col not in [lat_col, lon_col]]
    print(f"保留的属性列: {attribute_cols}")
    
    # 检查所有保留的列是否为数值类型
    numeric_cols = []
    for col in attribute_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].dropna().size > 0:  # 如果有有效数值
            numeric_cols.append(col)
    
    print(f"有效的数值列: {numeric_cols}")
    
    if not numeric_cols:
        print("错误: 没有找到有效的数值列")
        return False
    
    # 清理数据，移除无效值
    all_cols_needed = [lat_col, lon_col] + numeric_cols
    df_clean = df.dropna(subset=all_cols_needed)
    
    if df_clean.empty:
        print(f"错误: 处理后没有有效数据")
        return False
    
    print(f"处理后的有效数据点数量: {len(df_clean)}")
    
    # 计算坐标范围
    min_lon, max_lon = df_clean[lon_col].min(), df_clean[lon_col].max()
    min_lat, max_lat = df_clean[lat_col].min(), df_clean[lat_col].max()
    
    print(f"坐标范围 - 经度: {min_lon:.6f} 到 {max_lon:.6f}, 纬度: {min_lat:.6f} 到 {max_lat:.6f}")
    
    # 计算经纬度差值（用于10米像素）
    lat_factor = 111320 * np.cos(np.radians((min_lat + max_lat) / 2))
    lon_per_pixel = pixel_size / lat_factor
    lat_per_pixel = pixel_size / 110574
    
    # 计算栅格尺寸
    width = int((max_lon - min_lon) / lon_per_pixel) + 1
    height = int((max_lat - min_lat) / lat_per_pixel) + 1
    
    print(f"栅格尺寸: {width} x {height}")
    print(f"像素分辨率 - 经度方向: {lon_per_pixel:.8f} 度, 纬度方向: {lat_per_pixel:.8f} 度")
    
    # 为每个数值列创建单独的栅格文件
    success_count = 0
    for value_col in numeric_cols:
        print(f"\n处理列: {value_col}")
        
        # 创建栅格数组
        raster_array = np.full((height, width), np.nan, dtype=np.float32)
        
        # 将点数据映射到栅格
        for _, row in df_clean.iterrows():
            lon, lat, value = row[lon_col], row[lat_col], row[value_col]
            
            # 计算在栅格中的行列索引
            col = int((lon - min_lon) / lon_per_pixel)
            row_idx = int((max_lat - lat) / lat_per_pixel)  # 注意Y轴方向
            
            # 检查索引是否在范围内
            if 0 <= col < width and 0 <= row_idx < height:
                # 如果该位置已有值，可以选择覆盖或跳过
                # 这里选择覆盖
                raster_array[row_idx, col] = value
        
        # 设置仿射变换参数
        transform = from_origin(min_lon, max_lat, lon_per_pixel, lat_per_pixel)
        
        # 生成输出文件名
        base_name = Path(input_csv).stem
        output_tif = os.path.join(output_folder, f"{base_name}_{value_col}.tif")
        
        # 创建TIF文件（使用WGS84坐标系统）
        with rasterio.open(
            output_tif,
            'w',
            driver='GTiff',
            height=height,
            width=width,
            count=1,
            dtype=raster_array.dtype,
            crs='EPSG:4326',  # WGS84坐标系统
            transform=transform,
            nodata=np.nan
        ) as dst:
            dst.write(raster_array, 1)
        
        print(f"成功创建栅格文件: {output_tif}")
        print(f"栅格统计信息:")
        print(f"  - 有效值数量: {np.count_nonzero(~np.isnan(raster_array))}")
        if not np.all(np.isnan(raster_array)):
            print(f"  - 最小值: {np.nanmin(raster_array):.6f}")
            print(f"  - 最大值: {np.nanmax(raster_array):.6f}")
            print(f"  - 平均值: {np.nanmean(raster_array):.6f}")
        else:
            print(f"  - 栅格中没有有效数值")
        
        success_count += 1
    
    # 保存处理后的数据（包含所有保留的列）
    processed_csv_path = os.path.join(output_folder, f"{base_name}_processed.csv")
    df_clean.to_csv(processed_csv_path, index=False)
    print(f"\n处理后的数据已保存: {processed_csv_path}")
    
    print(f"\n总共成功创建了 {success_count} 个栅格文件")
    
    return True

# 设置输入文件路径
input_csv = r"E:\chuli\ratio_to_tif\ratio_NY_individual.csv"
output_folder = r"E:\chuli\ratio_to_tif"

# 执行转换
if os.path.exists(input_csv):
    success = csv_to_tif_multiple_attributes(input_csv, output_folder, pixel_size=10)
    if success:
        print(f"\n转换完成！")
        print(f"输出文件保存在: {output_folder}")
        print(f"创建的栅格文件:")
        for col in ['wall', 'building', 'sky', 'road', 'tree', 'fence', 
                    'sidewalk', 'pole', 'signboard', 'streetlight']:
            tif_file = os.path.join(output_folder, f"ratio_NY_individual_{col}.tif")
            if os.path.exists(tif_file):
                print(f"  - {col}.tif")
    else:
        print("转换失败！")
else:
    print(f"输入文件不存在: {input_csv}")

# 验证生成的TIF文件
print("\n验证生成的TIF文件:")
tif_files = list(Path(output_folder).glob(f"ratio_NY_individual_*.tif"))
tif_files = [f for f in tif_files if not f.name.endswith('_processed.csv.tif')]
for tif_file in tif_files:
    try:
        with rasterio.open(tif_file) as src:
            print(f"文件: {tif_file.name}")
            print(f"  - 尺寸: {src.width} x {src.height}")
            print(f"  - 波段数: {src.count}")
            print(f"  - 数据类型: {src.dtypes[0]}")
            print(f"  - 坐标系: {src.crs}")
            print(f"  - 范围: {src.bounds}")
            print("-" * 30)
    except Exception as e:
        print(f"读取文件 {tif_file} 时出错: {str(e)}")

# 显示处理后的数据预览
processed_file = os.path.join(output_folder, f"ratio_NY_individual_processed.csv")
if os.path.exists(processed_file):
    print(f"\n处理后数据预览:")
    processed_df = pd.read_csv(processed_file)
    print(processed_df.head())
    print(f"处理后数据形状: {processed_df.shape}")
    print(f"列名: {list(processed_df.columns)}")


In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def csv_to_tif_multiple_attributes(input_csv, output_folder, pixel_size=10):
    """
    将CSV文件转换为多个TIF栅格文件
    date和ratio_svid字段被删除，其他所有数值字段都输出为独立的栅格文件
    
    参数:
    input_csv: 输入CSV文件路径
    output_folder: 输出文件夹路径
    pixel_size: 像素大小（米）
    """
    # 读取CSV文件
    df = pd.read_csv(input_csv)
    print(f"正在处理文件: {input_csv}")
    print(f"CSV文件形状: {df.shape}")
    print(f"列名: {list(df.columns)}")
    
    # 确保输出文件夹存在
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # 识别经纬度列
    lat_col = 'lat'
    lon_col = 'lon'
    
    # 检查经纬度列是否存在
    if lat_col not in df.columns or lon_col not in df.columns:
        print(f"错误: 缺少必要的经纬度列 ({lat_col}, {lon_col})")
        return False
    
    # 删除date和ratio_svid列
    columns_to_drop = []
    if 'date' in df.columns:
        columns_to_drop.append('date')
    if 'ratio_svid' in df.columns:
        columns_to_drop.append('ratio_svid')
    
    if columns_to_drop:
        df = df.drop(columns=columns_to_drop)
        print(f"已删除列: {columns_to_drop}")
    
    # 检查坐标是否为数值类型
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # 获取除经纬度外的所有其他列（这些应该是数值列）
    attribute_cols = [col for col in df.columns if col not in [lat_col, lon_col]]
    print(f"保留的属性列: {attribute_cols}")
    
    # 检查所有保留的列是否为数值类型
    numeric_cols = []
    for col in attribute_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if df[col].dropna().size > 0:  # 如果有有效数值
            numeric_cols.append(col)
    
    print(f"有效的数值列: {numeric_cols}")
    
    if not numeric_cols:
        print("错误: 没有找到有效的数值列")
        return False
    
    # 清理数据，移除无效值
    all_cols_needed = [lat_col, lon_col] + numeric_cols
    df_clean = df.dropna(subset=all_cols_needed)
    
    if df_clean.empty:
        print(f"错误: 处理后没有有效数据")
        return False
    
    print(f"处理后的有效数据点数量: {len(df_clean)}")
    
    # 计算坐标范围
    min_lon, max_lon = df_clean[lon_col].min(), df_clean[lon_col].max()
    min_lat, max_lat = df_clean[lat_col].min(), df_clean[lat_col].max()
    
    print(f"坐标范围 - 经度: {min_lon:.6f} 到 {max_lon:.6f}, 纬度: {min_lat:.6f} 到 {max_lat:.6f}")
    
    # 计算经纬度差值（用于10米像素）
    lat_factor = 111320 * np.cos(np.radians((min_lat + max_lat) / 2))
    lon_per_pixel = pixel_size / lat_factor
    lat_per_pixel = pixel_size / 110574
    
    # 计算栅格尺寸
    width = int((max_lon - min_lon) / lon_per_pixel) + 1
    height = int((max_lat - min_lat) / lat_per_pixel) + 1
    
    print(f"栅格尺寸: {width} x {height}")
    print(f"像素分辨率 - 经度方向: {lon_per_pixel:.8f} 度, 纬度方向: {lat_per_pixel:.8f} 度")
    
    # 为每个数值列创建单独的栅格文件
    success_count = 0
    for value_col in numeric_cols:
        print(f"\n处理列: {value_col}")
        
        # 创建栅格数组
        raster_array = np.full((height, width), np.nan, dtype=np.float32)
        
        # 将点数据映射到栅格
        for _, row in df_clean.iterrows():
            lon, lat, value = row[lon_col], row[lat_col], row[value_col]
            
            # 计算在栅格中的行列索引
            col = int((lon - min_lon) / lon_per_pixel)
            row_idx = int((max_lat - lat) / lat_per_pixel)  # 注意Y轴方向
            
            # 检查索引是否在范围内
            if 0 <= col < width and 0 <= row_idx < height:
                # 如果该位置已有值，可以选择覆盖或跳过
                # 这里选择覆盖
                raster_array[row_idx, col] = value
        
        # 设置仿射变换参数
        transform = from_origin(min_lon, max_lat, lon_per_pixel, lat_per_pixel)
        
        # 生成输出文件名
        base_name = Path(input_csv).stem
        output_tif = os.path.join(output_folder, f"{base_name}_{value_col}.tif")
        
        # 创建TIF文件（使用WGS84坐标系统）
        with rasterio.open(
            output_tif,
            'w',
            driver='GTiff',
            height=height,
            width=width,
            count=1,
            dtype=raster_array.dtype,
            crs='EPSG:4326',  # WGS84坐标系统
            transform=transform,
            nodata=np.nan
        ) as dst:
            dst.write(raster_array, 1)
        
        print(f"成功创建栅格文件: {output_tif}")
        print(f"栅格统计信息:")
        print(f"  - 有效值数量: {np.count_nonzero(~np.isnan(raster_array))}")
        if not np.all(np.isnan(raster_array)):
            print(f"  - 最小值: {np.nanmin(raster_array):.6f}")
            print(f"  - 最大值: {np.nanmax(raster_array):.6f}")
            print(f"  - 平均值: {np.nanmean(raster_array):.6f}")
        else:
            print(f"  - 栅格中没有有效数值")
        
        success_count += 1
    
    # 保存处理后的数据（包含所有保留的列）
    processed_csv_path = os.path.join(output_folder, f"{base_name}_processed.csv")
    df_clean.to_csv(processed_csv_path, index=False)
    print(f"\n处理后的数据已保存: {processed_csv_path}")
    
    print(f"\n总共成功创建了 {success_count} 个栅格文件")
    
    return True

# 设置输入文件路径
input_csv = r"E:\chuli\ratio_to_tif\ratio_NY_plus.csv"
output_folder = r"E:\chuli\ratio_to_tif"

# 执行转换
if os.path.exists(input_csv):
    success = csv_to_tif_multiple_attributes(input_csv, output_folder, pixel_size=10)
    if success:
        print(f"\n转换完成！")
        print(f"输出文件保存在: {output_folder}")
        print(f"创建的栅格文件:")
        for col in ['Enclosure_degree', 'Openness', 'Protection', 'Surveillance_and_lighting', 
                    'Street_furniture', 'Greening_degree', 'Traffic_flow', 'Pedestrian_density']:
            tif_file = os.path.join(output_folder, f"ratio_NY_plus_{col}.tif")
            if os.path.exists(tif_file):
                print(f"  - {col}.tif")
    else:
        print("转换失败！")
else:
    print(f"输入文件不存在: {input_csv}")

# 验证生成的TIF文件
print("\n验证生成的TIF文件:")
tif_files = list(Path(output_folder).glob(f"ratio_NY_plus_*.tif"))
tif_files = [f for f in tif_files if not f.name.endswith('_processed.csv.tif')]
for tif_file in tif_files:
    try:
        with rasterio.open(tif_file) as src:
            print(f"文件: {tif_file.name}")
            print(f"  - 尺寸: {src.width} x {src.height}")
            print(f"  - 波段数: {src.count}")
            print(f"  - 数据类型: {src.dtypes[0]}")
            print(f"  - 坐标系: {src.crs}")
            print(f"  - 范围: {src.bounds}")
            print("-" * 30)
    except Exception as e:
        print(f"读取文件 {tif_file} 时出错: {str(e)}")

# 显示处理后的数据预览
processed_file = os.path.join(output_folder, f"ratio_NY_plus_processed.csv")
if os.path.exists(processed_file):
    print(f"\n处理后数据预览:")
    processed_df = pd.read_csv(processed_file)
    print(processed_df.head())
    print(f"处理后数据形状: {processed_df.shape}")
    print(f"列名: {list(processed_df.columns)}")


In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def csv_to_tif_with_attributes(input_csv, output_folder, pixel_size=10):
    """
    将CSV文件转换为TIF栅格文件，并为所有字段创建编码表
    DistrictCh和SchoolNa_1字段被删除，其他所有字段都进行编码保留
    
    参数:
    input_csv: 输入CSV文件路径
    output_folder: 输出文件夹路径
    pixel_size: 像素大小（米）
    """
    # 读取CSV文件
    df = pd.read_csv(input_csv)
    print(f"正在处理文件: {input_csv}")
    print(f"CSV文件形状: {df.shape}")
    print(f"列名: {list(df.columns)}")
    
    # 确保输出文件夹存在
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # 识别经纬度列
    lat_col = 'latitude'
    lon_col = 'longitude'
    
    # 检查经纬度列是否存在
    if lat_col not in df.columns or lon_col not in df.columns:
        print(f"错误: 缺少必要的经纬度列 ({lat_col}, {lon_col})")
        return False
    
    # 删除DistrictCh和SchoolNa_1列
    columns_to_drop = []
    if 'DistrictCh' in df.columns:
        columns_to_drop.append('DistrictCh')
    if 'SchoolNa_1' in df.columns:
        columns_to_drop.append('SchoolNa_1')
    
    if columns_to_drop:
        df = df.drop(columns=columns_to_drop)
        print(f"已删除列: {columns_to_drop}")
    
    # 检查坐标是否为数值类型
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # 获取除经纬度外的所有其他列
    attribute_cols = [col for col in df.columns if col not in [lat_col, lon_col]]
    print(f"保留的属性列: {attribute_cols}")
    
    # 创建编码字典
    encoding_dict = {}
    
    # 处理所有非经纬度列，创建编码
    for col in attribute_cols:
        unique_values = df[col].dropna().unique()
        value_to_code = {val: idx for idx, val in enumerate(unique_values)}
        encoding_dict[col] = value_to_code
        
        # 将原列转换为数值编码
        df[col] = df[col].map(value_to_code)
    
    # 创建编码表CSV文件
    if encoding_dict:
        encoding_csv_path = os.path.join(output_folder, f"{Path(input_csv).stem}_encoding.csv")
        encoding_data = []
        for col, mapping in encoding_dict.items():
            for original_value, code in mapping.items():
                encoding_data.append({
                    'Column': col,
                    'Original_Value': original_value,
                    'Encoded_Value': code
                })
        
        encoding_df = pd.DataFrame(encoding_data)
        encoding_df.to_csv(encoding_csv_path, index=False)
        print(f"编码表已保存: {encoding_csv_path}")
    
    # 选择一个字段作为栅格值（这里选择第一个可用字段）
    if attribute_cols:
        value_col = attribute_cols[0]
        print(f"使用列 '{value_col}' 作为栅格值")
    else:
        print(f"错误: 没有可用的属性列")
        return False
    
    # 清理数据，移除无效值
    df_clean = df.dropna(subset=[lat_col, lon_col, value_col])
    
    if df_clean.empty:
        print(f"错误: 处理后没有有效数据")
        return False
    
    print(f"处理后的有效数据点数量: {len(df_clean)}")
    
    # 计算坐标范围
    min_lon, max_lon = df_clean[lon_col].min(), df_clean[lon_col].max()
    min_lat, max_lat = df_clean[lat_col].min(), df_clean[lat_col].max()
    
    print(f"坐标范围 - 经度: {min_lon:.6f} 到 {max_lon:.6f}, 纬度: {min_lat:.6f} 到 {max_lat:.6f}")
    
    # 计算经纬度差值（用于10米像素）
    lat_factor = 111320 * np.cos(np.radians((min_lat + max_lat) / 2))
    lon_per_pixel = pixel_size / lat_factor
    lat_per_pixel = pixel_size / 110574
    
    # 计算栅格尺寸
    width = int((max_lon - min_lon) / lon_per_pixel) + 1
    height = int((max_lat - min_lat) / lat_per_pixel) + 1
    
    print(f"栅格尺寸: {width} x {height}")
    print(f"像素分辨率 - 经度方向: {lon_per_pixel:.8f} 度, 纬度方向: {lat_per_pixel:.8f} 度")
    
    # 创建栅格数组
    raster_array = np.full((height, width), np.nan, dtype=np.float32)
    
    # 将点数据映射到栅格
    for _, row in df_clean.iterrows():
        lon, lat, value = row[lon_col], row[lat_col], row[value_col]
        
        # 计算在栅格中的行列索引
        col = int((lon - min_lon) / lon_per_pixel)
        row_idx = int((max_lat - lat) / lat_per_pixel)  # 注意Y轴方向
        
        # 检查索引是否在范围内
        if 0 <= col < width and 0 <= row_idx < height:
            # 如果该位置已有值，可以选择覆盖或跳过
            # 这里选择覆盖
            raster_array[row_idx, col] = value
    
    # 设置仿射变换参数
    transform = from_origin(min_lon, max_lat, lon_per_pixel, lat_per_pixel)
    
    # 生成输出文件名
    base_name = Path(input_csv).stem
    output_tif = os.path.join(output_folder, f"{base_name}_grid.tif")
    
    # 创建TIF文件（使用WGS84坐标系统）
    with rasterio.open(
        output_tif,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=raster_array.dtype,
        crs='EPSG:4326',  # WGS84坐标系统
        transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(raster_array, 1)
    
    print(f"成功创建栅格文件: {output_tif}")
    print(f"栅格统计信息:")
    print(f"  - 有效值数量: {np.count_nonzero(~np.isnan(raster_array))}")
    if not np.all(np.isnan(raster_array)):
        print(f"  - 最小值: {np.nanmin(raster_array):.4f}")
        print(f"  - 最大值: {np.nanmax(raster_array):.4f}")
        print(f"  - 平均值: {np.nanmean(raster_array):.4f}")
    else:
        print(f"  - 栅格中没有有效数值")
    
    # 保存处理后的数据（包含编码后的所有字段）
    processed_csv_path = os.path.join(output_folder, f"{base_name}_processed.csv")
    df_clean.to_csv(processed_csv_path, index=False)
    print(f"处理后的数据已保存: {processed_csv_path}")
    
    return True

# 设置输入文件路径
input_csv = r"E:\chuli\ratio_to_tif\school_HK.csv"
output_folder = r"E:\chuli\ratio_to_tif"

# 执行转换
if os.path.exists(input_csv):
    success = csv_to_tif_with_attributes(input_csv, output_folder, pixel_size=10)
    if success:
        print(f"\n转换完成！")
        print(f"输出文件:")
        print(f"  - 栅格文件: {Path(input_csv).stem}_grid.tif")
        print(f"  - 编码表: {Path(input_csv).stem}_encoding.csv")
        print(f"  - 处理后数据: {Path(input_csv).stem}_processed.csv")
    else:
        print("转换失败！")
else:
    print(f"输入文件不存在: {input_csv}")

# 验证生成的TIF文件
print("\n验证生成的TIF文件:")
tif_files = list(Path(output_folder).glob(f"{Path(input_csv).stem}_grid.tif"))
for tif_file in tif_files:
    try:
        with rasterio.open(tif_file) as src:
            print(f"文件: {tif_file.name}")
            print(f"  - 尺寸: {src.width} x {src.height}")
            print(f"  - 波段数: {src.count}")
            print(f"  - 数据类型: {src.dtypes[0]}")
            print(f"  - 坐标系: {src.crs}")
            print(f"  - 范围: {src.bounds}")
            print("-" * 30)
    except Exception as e:
        print(f"读取文件 {tif_file} 时出错: {str(e)}")

# 显示编码表内容
encoding_file = os.path.join(output_folder, f"{Path(input_csv).stem}_encoding.csv")
if os.path.exists(encoding_file):
    print(f"\n编码表内容:")
    encoding_df = pd.read_csv(encoding_file)
    print(encoding_df)
else:
    print(f"\n未生成编码表（可能没有字符串字段需要编码）")

# 显示处理后的数据预览
processed_file = os.path.join(output_folder, f"{Path(input_csv).stem}_processed.csv")
if os.path.exists(processed_file):
    print(f"\n处理后数据预览:")
    processed_df = pd.read_csv(processed_file)
    print(processed_df.head())
    print(f"处理后数据形状: {processed_df.shape}")


In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def csv_to_tif_with_attributes(input_csv, output_folder, pixel_size=10):
    """
    将CSV文件转换为TIF栅格文件，并为所有字段创建编码表
    Name和location_code字段被删除，其他所有字段都进行编码保留
    
    参数:
    input_csv: 输入CSV文件路径
    output_folder: 输出文件夹路径
    pixel_size: 像素大小（米）
    """
    # 读取CSV文件
    df = pd.read_csv(input_csv)
    print(f"正在处理文件: {input_csv}")
    print(f"CSV文件形状: {df.shape}")
    print(f"列名: {list(df.columns)}")
    
    # 确保输出文件夹存在
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    # 识别经纬度列
    lat_col = 'Latitude'
    lon_col = 'Longitude'
    
    # 检查经纬度列是否存在
    if lat_col not in df.columns or lon_col not in df.columns:
        print(f"错误: 缺少必要的经纬度列 ({lat_col}, {lon_col})")
        return False
    
    # 删除Name和location_code列
    columns_to_drop = []
    if 'Name' in df.columns:
        columns_to_drop.append('Name')
    if 'location_code' in df.columns:
        columns_to_drop.append('location_code')
    
    if columns_to_drop:
        df = df.drop(columns=columns_to_drop)
        print(f"已删除列: {columns_to_drop}")
    
    # 检查坐标是否为数值类型
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # 获取除经纬度外的所有其他列
    attribute_cols = [col for col in df.columns if col not in [lat_col, lon_col]]
    print(f"保留的属性列: {attribute_cols}")
    
    # 创建编码字典
    encoding_dict = {}
    
    # 处理所有非经纬度列，创建编码
    for col in attribute_cols:
        unique_values = df[col].dropna().unique()
        value_to_code = {val: idx for idx, val in enumerate(unique_values)}
        encoding_dict[col] = value_to_code
        
        # 将原列转换为数值编码
        df[col] = df[col].map(value_to_code)
    
    # 创建编码表CSV文件
    if encoding_dict:
        encoding_csv_path = os.path.join(output_folder, f"{Path(input_csv).stem}_encoding.csv")
        encoding_data = []
        for col, mapping in encoding_dict.items():
            for original_value, code in mapping.items():
                encoding_data.append({
                    'Column': col,
                    'Original_Value': original_value,
                    'Encoded_Value': code
                })
        
        encoding_df = pd.DataFrame(encoding_data)
        encoding_df.to_csv(encoding_csv_path, index=False)
        print(f"编码表已保存: {encoding_csv_path}")
    
    # 选择一个字段作为栅格值（这里选择第一个可用字段）
    if attribute_cols:
        value_col = attribute_cols[0]
        print(f"使用列 '{value_col}' 作为栅格值")
    else:
        print(f"错误: 没有可用的属性列")
        return False
    
    # 清理数据，移除无效值
    df_clean = df.dropna(subset=[lat_col, lon_col, value_col])
    
    if df_clean.empty:
        print(f"错误: 处理后没有有效数据")
        return False
    
    print(f"处理后的有效数据点数量: {len(df_clean)}")
    
    # 计算坐标范围
    min_lon, max_lon = df_clean[lon_col].min(), df_clean[lon_col].max()
    min_lat, max_lat = df_clean[lat_col].min(), df_clean[lat_col].max()
    
    print(f"坐标范围 - 经度: {min_lon:.6f} 到 {max_lon:.6f}, 纬度: {min_lat:.6f} 到 {max_lat:.6f}")
    
    # 计算经纬度差值（用于10米像素）
    lat_factor = 111320 * np.cos(np.radians((min_lat + max_lat) / 2))
    lon_per_pixel = pixel_size / lat_factor
    lat_per_pixel = pixel_size / 110574
    
    # 计算栅格尺寸
    width = int((max_lon - min_lon) / lon_per_pixel) + 1
    height = int((max_lat - min_lat) / lat_per_pixel) + 1
    
    print(f"栅格尺寸: {width} x {height}")
    print(f"像素分辨率 - 经度方向: {lon_per_pixel:.8f} 度, 纬度方向: {lat_per_pixel:.8f} 度")
    
    # 创建栅格数组
    raster_array = np.full((height, width), np.nan, dtype=np.float32)
    
    # 将点数据映射到栅格
    for _, row in df_clean.iterrows():
        lon, lat, value = row[lon_col], row[lat_col], row[value_col]
        
        # 计算在栅格中的行列索引
        col = int((lon - min_lon) / lon_per_pixel)
        row_idx = int((max_lat - lat) / lat_per_pixel)  # 注意Y轴方向
        
        # 检查索引是否在范围内
        if 0 <= col < width and 0 <= row_idx < height:
            # 如果该位置已有值，可以选择覆盖或跳过
            # 这里选择覆盖
            raster_array[row_idx, col] = value
    
    # 设置仿射变换参数
    transform = from_origin(min_lon, max_lat, lon_per_pixel, lat_per_pixel)
    
    # 生成输出文件名
    base_name = Path(input_csv).stem
    output_tif = os.path.join(output_folder, f"{base_name}_grid.tif")
    
    # 创建TIF文件（使用WGS84坐标系统）
    with rasterio.open(
        output_tif,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=raster_array.dtype,
        crs='EPSG:4326',  # WGS84坐标系统
        transform=transform,
        nodata=np.nan
    ) as dst:
        dst.write(raster_array, 1)
    
    print(f"成功创建栅格文件: {output_tif}")
    print(f"栅格统计信息:")
    print(f"  - 有效值数量: {np.count_nonzero(~np.isnan(raster_array))}")
    if not np.all(np.isnan(raster_array)):
        print(f"  - 最小值: {np.nanmin(raster_array):.4f}")
        print(f"  - 最大值: {np.nanmax(raster_array):.4f}")
        print(f"  - 平均值: {np.nanmean(raster_array):.4f}")
    else:
        print(f"  - 栅格中没有有效数值")
    
    # 保存处理后的数据（包含编码后的所有字段）
    processed_csv_path = os.path.join(output_folder, f"{base_name}_processed.csv")
    df_clean.to_csv(processed_csv_path, index=False)
    print(f"处理后的数据已保存: {processed_csv_path}")
    
    return True

# 设置输入文件路径
input_csv = r"E:\chuli\ratio_to_tif\school_NY.csv"
output_folder = r"E:\chuli\ratio_to_tif"

# 执行转换
if os.path.exists(input_csv):
    success = csv_to_tif_with_attributes(input_csv, output_folder, pixel_size=10)
    if success:
        print(f"\n转换完成！")
        print(f"输出文件:")
        print(f"  - 栅格文件: {Path(input_csv).stem}_grid.tif")
        print(f"  - 编码表: {Path(input_csv).stem}_encoding.csv")
        print(f"  - 处理后数据: {Path(input_csv).stem}_processed.csv")
    else:
        print("转换失败！")
else:
    print(f"输入文件不存在: {input_csv}")

# 验证生成的TIF文件
print("\n验证生成的TIF文件:")
tif_files = list(Path(output_folder).glob(f"{Path(input_csv).stem}_grid.tif"))
for tif_file in tif_files:
    try:
        with rasterio.open(tif_file) as src:
            print(f"文件: {tif_file.name}")
            print(f"  - 尺寸: {src.width} x {src.height}")
            print(f"  - 波段数: {src.count}")
            print(f"  - 数据类型: {src.dtypes[0]}")
            print(f"  - 坐标系: {src.crs}")
            print(f"  - 范围: {src.bounds}")
            print("-" * 30)
    except Exception as e:
        print(f"读取文件 {tif_file} 时出错: {str(e)}")

# 显示编码表内容
encoding_file = os.path.join(output_folder, f"{Path(input_csv).stem}_encoding.csv")
if os.path.exists(encoding_file):
    print(f"\n编码表内容:")
    encoding_df = pd.read_csv(encoding_file)
    print(encoding_df)
else:
    print(f"\n未生成编码表（可能没有字符串字段需要编码）")

# 显示处理后的数据预览
processed_file = os.path.join(output_folder, f"{Path(input_csv).stem}_processed.csv")
if os.path.exists(processed_file):
    print(f"\n处理后数据预览:")
    processed_df = pd.read_csv(processed_file)
    print(processed_df.head())
    print(f"处理后数据形状: {processed_df.shape}")
